# UAV Leader-Follower Swarms / Optical Relative Navigation — tech-intel pipeline

Clone of the `techintel-jamming` OpenAlex pipeline, re-run on a **Boolean keyword Works corpus**
for UAV swarms and leader-follower formations that position, navigate and synchronise off each
other, with emphasis on optical links (infrared, visible light, ultraviolet / solar-blind, LED
beacons). Produces the three frames plus the two lookup dicts consumed by `leaderfollower2d.py`.

Run with the pipeline environment (`requirements-pipeline.txt`, Python 3.12), then run
`resave_pickles.py` with the app environment (Python 3.11, pandas 2.0.0).

In [1]:
import os, json, time, pickle, textwrap
import requests
import pyalex
from pyalex import Works
import pandas as pd
import numpy as np

# OpenAlex: keyless requests share a small daily budget per IP; a free key
# (https://openalex.org/settings/api) raises it. Set OPENALEX_API_KEY to use one.
OPENALEX_API_KEY = os.environ.get("OPENALEX_API_KEY")
if OPENALEX_API_KEY:
    pyalex.config.api_key = OPENALEX_API_KEY
pyalex.config.max_retries = 6
pyalex.config.retry_backoff_factor = 1.5
pyalex.config.retry_http_codes = [429, 500, 503]

from flair.embeddings import DocumentPoolEmbeddings
from flair.data import Sentence
from flair.embeddings import SentenceTransformerDocumentEmbeddings

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
DOC_EMBEDDINGS = DocumentPoolEmbeddings([SentenceTransformerDocumentEmbeddings(EMBEDDING_MODEL)])

import torch
from tqdm import tqdm
import yake
import umap.umap_ as umap

/home/david/jobs/openalex_local/techintel-jamming/.venv/lib/python3.12/site-packages/flair/embeddings/document.py:591: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


## 1. Corpus — Boolean OpenAlex `Works` search (title + abstract)

Inter-UAV optical positioning is not an OpenAlex *topic* of its own, so the corpus is the union of
three Boolean searches over title + abstract (not full text), deduplicated by paper id:

* **A — optical swarm PNT**: UAV × swarm/formation × infrared / visible light / ultraviolet / beacon × positioning.
* **C — leader-follower**: UAV × "leader-follower".
* **D — relative localization**: UAV × swarm/formation × relative / mutual / cooperative localization or navigation.

A is the on-target niche; C and D supply the surrounding field (UWB, vision and RF alternatives,
formation control) so the optical work can be seen in context. No country filter.

In [2]:
UAV = '("UAV" OR "drone" OR "unmanned aerial" OR "quadrotor" OR "MAV")'
SWARM = '("swarm" OR "leader-follower" OR "multi-UAV" OR "formation" OR "cooperative")'

QUERIES = {
    "A_optical_swarm_pnt":
        UAV
        + ' AND ("swarm" OR "leader-follower" OR "multi-UAV" OR "formation" OR "cooperative" OR "master" OR "anchor")'
        + ' AND ("infrared" OR "visible light" OR "VLC" OR "VLP" OR "ultraviolet" OR "solar-blind" OR "UV"'
          ' OR "optical wireless" OR "LED" OR "beacon")'
        + ' AND ("positioning" OR "localization" OR "navigation" OR "pose" OR "PNT"'
          ' OR "time synchronization" OR "ranging")',
    "C_leader_follower":
        '("UAV" OR "drone" OR "unmanned aerial" OR "quadrotor") AND ("leader-follower" OR "leader follower")',
    "D_relative_localization":
        UAV + ' AND ' + SWARM
        + ' AND ("relative localization" OR "relative positioning" OR "mutual localization"'
          ' OR "cooperative navigation" OR "cooperative localization" OR "relative navigation")',
}
# safety cap per query (none approaches this; keeps the pull bounded)
N_MAX_PER_QUERY = 3000
# raw pull is cached here (gitignored) so re-running the notebook does not re-spend API budget
RAW_CACHE = "cache/works_raw.pkl"

In [3]:
def process_works_list(worklist:list):
    '''
    transforms the
    works list into a dataframe.
    '''
    abstracts_dict = {h["id"]:h["abstract"] for h in worklist}
    df = pd.DataFrame.from_records(worklist)
    df.drop(columns=['abstract_inverted_index'], errors='ignore', inplace=True)
    df['abstract'] = df['id'].map(abstracts_dict)
    return df


def get_search_frame(tag:str, query:str, n_max:int=N_MAX_PER_QUERY):
    '''
    Boolean Works search for `query` over title+abstract, paginated and
    deduped by paper id. OpenAlex limits heavy Boolean searches to one a
    second, hence the pause between pages. Cursor paging needs an API key;
    without one, page-number paging is used (good for up to 10,000 results).
    '''
    pager = Works().filter(
        publication_year='>2000',
        title_and_abstract={"search": query},
    ).paginate(method="cursor" if OPENALEX_API_KEY else "page",
               per_page=200, n_max=n_max)
    frames = []
    for page in tqdm(pager, desc=tag):
        if page:
            frames.append(process_works_list(page))
        time.sleep(1.2)
    df = pd.concat(frames, ignore_index=True)
    df.drop_duplicates(subset='id', keep='first', inplace=True)
    df['query_tag'] = tag
    return df

In [4]:
if os.path.exists(RAW_CACHE):
    dftop = pd.read_pickle(RAW_CACHE)
    print("loaded cached raw pull:", dftop.shape)
else:
    frames_list = []
    for tag, query in QUERIES.items():
        df = get_search_frame(tag, query)
        print(f"{tag}: {len(df)} papers")
        frames_list.append(df)
    dftop = pd.concat(frames_list, ignore_index=True)
    os.makedirs(os.path.dirname(RAW_CACHE), exist_ok=True)
    dftop.to_pickle(RAW_CACHE)
print(dftop.groupby('query_tag')['id'].nunique())

loaded cached raw pull: (2730, 62)
query_tag
A_optical_swarm_pnt         570
C_leader_follower          1041
D_relative_localization    1119
Name: id, dtype: int64


In [5]:
# which queries matched each paper (kept as a column for later inspection)
query_tags = dftop.groupby('id')['query_tag'].apply(lambda s: sorted(set(s)))

dftop = dftop.drop_duplicates(subset='id', keep='first').copy()
dftop.set_index('id', inplace=True, drop=False)
dftop['query_tags'] = query_tags

dfall = dftop
print("corpus shape:", dfall.shape)

dfall['content'] = dfall['title'] + ". " + dfall['abstract']

dfrecords = dfall[~dfall['content'].isna()].copy()
print("with content:", dfrecords.shape)

# OpenAlex no longer always returns a 'grants' field (and 'locations' can be
# absent for sparse records); guarantee the columns the pipeline selects on
# exist, defaulting to an empty list per row.
for _col in ['grants', 'locations', 'concepts']:
    if _col not in dfrecords.columns:
        dfrecords[_col] = [[] for _ in range(len(dfrecords))]
    else:
        dfrecords[_col] = dfrecords[_col].apply(
            lambda v: v if isinstance(v, list) else [])

corpus shape: (2599, 63)
with content: (2216, 64)


## 2. Keywords + top concepts

In [6]:
def get_keywords(text:str, top:int=7, stopwords=None):
    '''
    takes a blob of text and
    returns the top **top**
    keywords as a list
    '''
    kw_extractor = yake.KeywordExtractor(top=top, stopwords=stopwords)
    keywords = kw_extractor.extract_keywords(text)
    return [p[0] for p in keywords]


def get_top_concepts(concept_list:list, score:float=.6):
    '''
    takes a list of concept dictionaries
    returns the display_names of concepts whose score is >= score
    '''
    return [c['display_name'] for c in concept_list if c['score'] >= score]

dfrecords['keywords'] = dfrecords['content'].apply(get_keywords)
dfrecords['top_concepts'] = dfrecords['concepts'].apply(get_top_concepts)

## 3. Sentence embeddings → UMAP (2D) → HDBSCAN clusters

In [7]:
def get_content_embeddings(dfrecords:pd.DataFrame) -> pd.DataFrame:
    '''
    passes the preprocessed content strings through the embedding model
    to produce the vector-space representation of each paper.
    '''
    sent = Sentence("The grass is green.")
    DOC_EMBEDDINGS.embed(sent)
    texts = dfrecords["content"].str.lower().values.tolist()
    all_descriptions = np.empty((len(texts), len(sent.embedding)))
    for i in tqdm(range(len(texts))):
        sent = Sentence(texts[i])
        DOC_EMBEDDINGS.embed(sent)
        all_descriptions[i, :] = sent.embedding.cpu().numpy()
    dfcontentvectors = pd.DataFrame.from_records(all_descriptions, index=dfrecords.index)
    return dfcontentvectors

# cached (gitignored) so a re-run is quick and gives identical vectors
VEC_CACHE = "cache/content_vectors.pkl"
if os.path.exists(VEC_CACHE) and pd.read_pickle(VEC_CACHE).index.equals(dfrecords.index):
    dfcontentvectors = pd.read_pickle(VEC_CACHE)
    print("loaded cached embeddings:", dfcontentvectors.shape)
else:
    dfcontentvectors = get_content_embeddings(dfrecords)
    dfcontentvectors.to_pickle(VEC_CACHE)

loaded cached embeddings: (2216, 384)


### Relevance filter

Short Boolean terms ("formation", "master", "LED", "UV", "pose") admit off-topic papers: geological
formations, crop imaging, wildfire detection, and stray matches on long abstracts. Each paper is
scored by its closest cosine similarity to a few sentences describing the subject, and papers below
`MIN_RELEVANCE` are removed before clustering. The cut was chosen by reading titles either side of
it; the removed papers are listed in `cache/removed_low_relevance.csv`.

In [8]:
TOPIC_SEEDS = [
    "leader-follower formation control of multiple unmanned aerial vehicles",
    "cooperative relative localization and navigation for a uav swarm in gnss-denied environments",
    "mutual localization of drones using ultraviolet, infrared or visible light led markers and beacons",
    "multi-uav swarm communication, coordination and cooperative path planning",
]
MIN_RELEVANCE = 0.37

def _embed(text):
    sent = Sentence(text)
    DOC_EMBEDDINGS.embed(sent)
    return sent.embedding.cpu().numpy()

_seeds = np.vstack([_embed(t) for t in TOPIC_SEEDS])
_seeds /= np.linalg.norm(_seeds, axis=1, keepdims=True)
_docs = dfcontentvectors.to_numpy()
_docs = _docs / np.linalg.norm(_docs, axis=1, keepdims=True)
dfrecords['relevance'] = (_docs @ _seeds.T).max(axis=1)

_removed = dfrecords[dfrecords['relevance'] < MIN_RELEVANCE]
_removed[['id', 'title', 'relevance', 'query_tags']].sort_values('relevance').to_csv(
    "cache/removed_low_relevance.csv", index=False)
print(f"removed {len(_removed)} low-relevance papers; {len(dfrecords) - len(_removed)} remain")
print(_removed['query_tags'].explode().value_counts().to_dict())
for t in _removed.sort_values('relevance', ascending=False)['title'].head(8):
    print("   just below the cut -", str(t)[:100])

dfrecords = dfrecords[dfrecords['relevance'] >= MIN_RELEVANCE].copy()
dfcontentvectors = dfcontentvectors.loc[dfrecords.index]

removed 263 low-relevance papers; 1953 remain
{'A_optical_swarm_pnt': 213, 'C_leader_follower': 29, 'D_relative_localization': 25}
   just below the cut - Decorrelation Method for Passive Interference Signals in the Selection System of Slow-Moving Targets
   just below the cut - Autonomous Earthwork Machinery for Urban Construction: A Review of Integrated Control, Fleet Coordin
   just below the cut - TEAMER: Electrically Engaged Undulation System for Unmanned Underwater Vehicles
   just below the cut - DNN-Based 3D Relative Positioning Using Inter-Node Distances in Indoor Environments
   just below the cut - Küresel Uçangöz Ekosisteminin Dönüşümü: Otonomi, Yapay Zekâ ve Pazar Stratejileri Üzerine Sistematik
   just below the cut - Gaze-DETR: Top-Down Guidance Through Priority Maps for Infrared Weak-Small UAV Detection with DETR
   just below the cut - Smart-Contract-based Trusted Task Offloading for UAV-Assisted Communications
   just below the cut - Improving RF Localization Through 

In [9]:
N_COMPONENTS = 2  # can visualize this way
umap_reducer = umap.UMAP(n_components=N_COMPONENTS,
                         random_state=1234,
                         metric='cosine')

reduced_vectors = umap_reducer.fit_transform(dfcontentvectors.to_numpy())
dfreduced = pd.DataFrame.from_records(reduced_vectors,
                index=dfcontentvectors.index)
dfreduced.columns = ['x','y']
dfreduced = dfreduced.astype('float64')

/home/david/jobs/openalex_local/techintel-jamming/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [10]:
import hdbscan

# This corpus is one continuous field rather than well-separated subjects, so
# the jamming settings (eom, epsilon 0.1) merge nearly everything into a single
# cluster. 'leaf' selection keeps the fine-grained topics.
hdbscan_args = {'min_cluster_size': 25,
                'min_samples': 5,
                'metric': 'euclidean',
                'cluster_selection_method': 'leaf',
                'prediction_data': True}

cluster = hdbscan.HDBSCAN(**hdbscan_args).fit(dfreduced[['x','y']].to_numpy())

# Leaf selection leaves nearly half the papers unassigned, and the app hides
# unassigned papers. Give each of those the cluster of its nearest assigned
# neighbours on the 2D map (majority of 5), scored by its HDBSCAN soft
# membership in that cluster, so it stays visible but carries little weight in
# the per-topic rankings (which sum this score).
from sklearn.neighbors import KNeighborsClassifier
xy = dfreduced[['x','y']].to_numpy()
labels = cluster.labels_.copy()
scores = cluster.probabilities_.copy()
unassigned = labels < 0
knn = KNeighborsClassifier(n_neighbors=5).fit(xy[~unassigned], labels[~unassigned])
labels[unassigned] = knn.predict(xy[unassigned])
soft = hdbscan.all_points_membership_vectors(cluster)
scores[unassigned] = soft[unassigned, labels[unassigned]]

dfreduced['cluster'] = labels
dfreduced['probability'] = scores
dfreduced['core_member'] = ~unassigned

dfpapers = dfrecords.merge(dfreduced, left_index=True,
                           right_index=True)
print("clusters (excl noise):", (dfreduced['cluster'].nunique() - (1 if -1 in dfreduced['cluster'].values else 0)))
print("core members:", int((~unassigned).sum()), "| soft-assigned:", int(unassigned.sum()), "of", len(dfreduced))

clusters (excl noise): 20
core members: 1018 | soft-assigned: 935 of 1953


### Topic listing (for inspection)

In [11]:
OPTICAL = r"(?i)ultraviolet|\\bUV\\b|solar-blind|visible light|\\bVLC\\b|\\bVLP\\b|infrared|\\bLEDs?\\b|beacon|optical"
for cl, g in dfpapers.groupby('cluster'):
    tags = pd.Series([t for ts in g['query_tags'] for t in ts]).value_counts().to_dict()
    print(f"\n=== cluster {cl}: {len(g)} papers ({int(g['core_member'].sum())} core)  {tags}")
    print("    optical terms in", f"{g['content'].str.contains(OPTICAL, regex=True).mean():.0%}", "of papers")
    for t in g.sort_values('probability', ascending=False)['title'].head(6):
        print("   -", str(t)[:110])


=== cluster 0: 36 papers (30 core)  {'D_relative_localization': 27, 'C_leader_follower': 7, 'A_optical_swarm_pnt': 3}
    optical terms in 8% of papers
   - Cooperative Multi-UAV Navigation in Complex Environments via Systematic Multi-Agent Deep Reinforcement Learnin
   - VACNA: Visibility-Aware Cooperative Navigation With Application in Inventory Management
   - Learning Emergence of Interaction Patterns across Independent RL Agents in Multi-Agent Environments
   - Eco-Friendly Disaster Evacuation Framework for 6G Connected and Autonomous Vehicular Networks
   - Prioritized Experience Replay in Multi-Actor-Attention-Critic for Reinforcement Learning
   - A deep multi-agent reinforcement learning framework for autonomous aerial navigation to grasping points on loa

=== cluster 1: 73 papers (46 core)  {'C_leader_follower': 55, 'D_relative_localization': 18, 'A_optical_swarm_pnt': 3}
    optical terms in 0% of papers
   - Towards Reliable Aerial Ground Vehicle Collaboration: An Integrat

    optical terms in 6% of papers
   - How to deal with low performance IMUs in an integrated navigation system: Step by step
   - sUAS Swarm Navigation using Inertial, Range Radios and Partial GNSS
   - LAWFIELD-CORE/INT: Hardware-Enforced Autonomous Swarm Governance (GPS-Intermittent Operations) - Technical Whi
   - Minimizing Relative Position and Attitude Error in UAV Swarms via General Form Consensus
   - Sensitivity analysis of a relative navigation solution for unmanned aerial vehicles in a GNSS-denied environme
   - Efficient Re-Propagation for Cooperative Navigation Under Delayed Measurements: A Galilean Group Approach

=== cluster 19: 79 papers (26 core)  {'D_relative_localization': 74, 'C_leader_follower': 5, 'A_optical_swarm_pnt': 4}
    optical terms in 6% of papers
   - Outlier-tolerant relative positioning method based on multi-source information fusion for unmanned aerial vehi
   - Research on collaborative multi-UAV localization method based on combination navigation i

## 4. Explode authorships → paper/author/affiliation triples

In [12]:
del dfpapers['id']
dfstart = dfpapers.reset_index()
dfbig = dfstart.explode(column='authorships')
dfbig.shape, dfstart.shape

((7101, 72), (1953, 72))

In [13]:
def add_extra_to_authorships(row: pd.DataFrame):
    '''
    row[authorships] is a dictionary;
    add in the paper-level keys to that dictionary.
    '''
    complete_dict = row["authorships"]
    if type(complete_dict) == dict:
        complete_dict["id"] = row["id"]
        complete_dict["x"] = row["x"]
        complete_dict["y"] = row["y"]
        complete_dict["cluster"] = row["cluster"]
        complete_dict["cluster_score"] = row["probability"]
        complete_dict["title"] = row["title"]
        complete_dict["abstract"] = row["abstract"]
        complete_dict["doi"] = row["doi"]
        complete_dict["publication_date"] = row["publication_date"]
        complete_dict["publication_year"] = row["publication_year"]
        complete_dict["grants"] = row["grants"]
        complete_dict["locations"] = row["locations"]
        # OpenAlex now returns raw_affiliation_strings (a list) and
        # is_corresponding; the app reads the older column names.
        complete_dict["raw_affiliation_string"] = "; ".join(
            complete_dict.get("raw_affiliation_strings") or [])
        complete_dict["is_corrresponding"] = complete_dict.get("is_corresponding")
        return complete_dict
    else:
        return row["authorships"]

dfbig['big_authorships'] = dfbig.apply(add_extra_to_authorships, axis=1)
bigvals = dfbig['authorships'].tolist()
dictvals = [c for c in bigvals if type(c) != float]

In [14]:
dftriple = pd.json_normalize(dictvals,
                  record_path=['institutions'],
                  meta=['id','raw_affiliation_string','author_position', 'doi',
                        'title','abstract','publication_date', 'publication_year',
                        'grants','locations',
                        'is_corrresponding','x','y','cluster','cluster_score',
                       ['author','id'], ['author', 'display_name'],
                       ['author','orcid']],
                  errors='ignore',
                  sep='_',
                  meta_prefix='paper_',
                 )
dftriple.shape

(7007, 24)

## 5. Topic centroids (keywords + concepts per cluster)

In [15]:
def get_cluster_concepts(topic_num:int, n:int=20):
    '''
    returns the list of top n occurring concepts from the top_concepts field
    for the given cluster.
    '''
    top_concepts = dfpapers[dfpapers['cluster'] == topic_num]['top_concepts'].tolist()
    flat_concepts = [item for sublist in top_concepts for item in sublist]
    concepts_dict = {c:flat_concepts.count(c) for c in flat_concepts}
    sorted_concepts = sorted(concepts_dict.items(), key=lambda x:x[1], reverse=True)
    return [c[0] for c in sorted_concepts][:n]


def get_yake_cluster_phrases(topic_num:int, n:int=20):
    '''
    returns the list of YAKE keyphrases for the concatenated abstracts of a cluster.
    '''
    documents = dfpapers[dfpapers['cluster'] == topic_num]['content'].tolist()
    topic_input = ". ".join(documents)
    kw_extractor = yake.KeywordExtractor(top=n, stopwords=None)
    keywords = kw_extractor.extract_keywords(topic_input)
    return [p[0] for p in keywords]


dfpapers['id'] = dfpapers.index
centroids = dfpapers.groupby('cluster')[['x','y']].mean().copy()
centroids['cluster'] = centroids.index
centroids['concepts'] = centroids['cluster'].apply(get_cluster_concepts)
centroids['keywords'] = centroids['cluster'].apply(get_yake_cluster_phrases)
centroids = centroids[['x','y','concepts','cluster','keywords']]

def wrap_it(x):
    return "<br>".join(textwrap.wrap(x, width=40))

centroids['wrapped_keywords'] = centroids['keywords'].apply(str).apply(wrap_it)
centroids['wrapped_concepts'] = centroids['concepts'].apply(str).apply(wrap_it)

## 6. Enrich dfinfo and dftriple (source / funders / affil & author lists)

In [16]:
def _primary_source(loc_list):
    try:
        return loc_list[0]["source"] or {}
    except Exception:
        return {}

def get_source_name(loc_list):
    return _primary_source(loc_list).get("display_name")

def get_source_type(loc_list):
    return _primary_source(loc_list).get("type")

def get_source_id(loc_list):
    return _primary_source(loc_list).get("id")

def get_funder_names(funder_list):
    '''
    funder_list is a list of grant dicts; return unique funder_display_name values.
    '''
    try:
        return list(set([f['funder_display_name'] for f in funder_list]))
    except:
        return []

dftriple["source"] = dftriple["paper_locations"].apply(get_source_name)
dftriple["source_type"] = dftriple["paper_locations"].apply(get_source_type)
dftriple["funder_list"] = dftriple["paper_grants"].apply(get_funder_names)

In [17]:
# full dfinfo (indexed by paper id) with everything leaderfollower2d.py reads
dfinfo = dfpapers[['x','y','id','title','abstract', 'doi','cluster','probability',
                 'publication_date','grants','locations',
                   'keywords','top_concepts']].copy()

dfinfo['affil_list'] = dftriple.groupby('paper_id')['paper_raw_affiliation_string'].\
    apply(lambda x: x.tolist())
dfinfo['author_list'] = dftriple.groupby('paper_id')['paper_author_display_name'].\
    apply(lambda x: x.tolist())

dfinfo['wrapped_affil_list'] = dfinfo['affil_list'].apply(str).apply(wrap_it)
dfinfo['wrapped_author_list'] = dfinfo['author_list'].apply(str).apply(wrap_it)
dfinfo['wrapped_keywords'] = dfinfo['keywords'].apply(str).apply(wrap_it)

dfinfo["source"] = dfinfo["locations"].apply(get_source_name)
dfinfo["source_type"] = dfinfo["locations"].apply(get_source_type)
dfinfo["funder_list"] = dfinfo["grants"].apply(get_funder_names)
dfinfo["wrapped_funder_list"] = dfinfo["funder_list"].apply(str).apply(wrap_it)

## 7. Write the three frames

The pipeline environment runs a newer pandas than the app (pandas 2.0.0), and pickles do not cross
that gap. So the frames are written here as JSON-lines, and `resave_pickles.py` (run with the app
environment) turns them into the `.pkl.gz` files the app loads.

In [18]:
os.makedirs("cache", exist_ok=True)
FRAMES = {"leaderfollowercentroids2d": centroids,
          "leaderfollowerdftriple2d": dftriple,
          "leaderfollowerdfinfo2d": dfinfo}
for name, frame in FRAMES.items():
    out = frame.copy()
    out.insert(0, "__index__", out.index)
    out.to_json(f"cache/{name}.jsonl", orient="records", lines=True, double_precision=15)
    out.dtypes.astype(str).to_json(f"cache/{name}.dtypes.json")
print("wrote centroids", centroids.shape, "| dftriple", dftriple.shape, "| dfinfo", dfinfo.shape)

wrote centroids (20, 7) | dftriple (7007, 27) | dfinfo (1953, 22)


## 8. Source homepage dict + affiliation geo dict

Looked up by exact OpenAlex **id**, 50 per request (a name search per entity is both less accurate
and far more requests). The dicts stay keyed by display_name because that is how the app reads them.

In [19]:
def fetch_by_ids(entity:str, ids:list, select:str, batch:int=50):
    '''returns {openalex id: record} for the given ids, fetched `batch` at a time.
    Results are cached in cache/<entity>.json; only ids not yet cached are requested.'''
    cache_file = f"cache/{entity}.json"
    out = json.load(open(cache_file)) if os.path.exists(cache_file) else {}
    ids = sorted({i for i in ids if isinstance(i, str) and i} - set(out))
    for k in tqdm(range(0, len(ids), batch), desc=entity):
        chunk = [i.rsplit("/", 1)[-1] for i in ids[k:k + batch]]
        params = {"filter": "openalex:" + "|".join(chunk), "per_page": batch, "select": select}
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY
        for attempt in range(5):
            r = requests.get(f"https://api.openalex.org/{entity}", params=params, timeout=60)
            if r.status_code == 200:
                break
            time.sleep(2 * (attempt + 1))
        r.raise_for_status()
        for rec in r.json()["results"]:
            out[rec["id"]] = rec
        time.sleep(0.2)
    with open(cache_file, "w") as f:
        json.dump(out, f)
    return out

# sources
src = pd.DataFrame({"name": dfinfo["source"], "id": dfinfo["locations"].apply(get_source_id)}).dropna()
src = src.drop_duplicates("name")
src_recs = fetch_by_ids("sources", src["id"].tolist(), "id,display_name,homepage_url")
source_page_dict = {n: (src_recs.get(i) or {}).get("homepage_url") for n, i in zip(src["name"], src["id"])}
print("sources:", len(source_page_dict), "with homepage:", sum(v is not None for v in source_page_dict.values()))
with open("updatesource_page_dict.pkl", "wb") as f:
    pickle.dump(source_page_dict, f)

sources: 0it [00:00, ?it/s]

sources: 0it [00:00, ?it/s]

sources: 786 with homepage: 279


In [20]:
# institutions
inst = dftriple[["display_name", "id"]].dropna().drop_duplicates("display_name")
inst_recs = fetch_by_ids("institutions", inst["id"].tolist(), "id,display_name,geo")
affil_geo_dict = {}
for n, i in zip(inst["display_name"], inst["id"]):
    geo = (inst_recs.get(i) or {}).get("geo") or {}
    affil_geo_dict[n] = (geo.get("latitude"), geo.get("longitude"))
_ok = sum(v[0] is not None for v in affil_geo_dict.values())
print(f"institutions: {len(affil_geo_dict)}, with geo: {_ok} ({_ok / max(len(affil_geo_dict), 1):.1%})")
with open("updateaffil_geo_dict.pkl", "wb") as f:
    pickle.dump(affil_geo_dict, f)
print("done.")

institutions: 0it [00:00, ?it/s]

institutions: 0it [00:00, ?it/s]

institutions: 1090, with geo: 1090 (100.0%)
done.
